In [1]:
import pandas as pd
import os

In [ ]:
# 분석에 사용할 원본 CSV 파일 경로를 지정
file_1 = "../data/대구광역시 북구_불법주정차위반정보_20230704.csv"
file_2 = "../data/대구광역시 북구_불법주정차위반정보_20231130.csv"
file_3 = "../data/대구광역시 북구_불법주정차위반정보_20240603.csv"

In [9]:
# CSV 파일 3개를 UTF-8 인코딩으로 불러옵니다.
df_1 = pd.read_csv(file_1, encoding="utf-8")
df_2 = pd.read_csv(file_2, encoding="utf-8")
df_3 = pd.read_csv(file_3, encoding="utf-8")

# 데이터가 정상적으로 불러와졌는지 확인합니다.
print(df_1.head())
print(df_2.head())
print(df_3.head())

      구분        위반일자   위반시간           위반장소     과태료부과일자     데이터기준일자
0  불법주정차  2017-01-01  11:08    태전1동 관음타운상가  2017-01-02  2023-07-04
1  불법주정차  2017-01-01  12:42      대현동 동대구시장  2017-01-02  2023-07-04
2  불법주정차  2017-01-01  14:24  침산동 건영하이츠(본죽)  2017-01-02  2023-07-04
3  불법주정차  2017-01-01  14:59  침산동 건영하이츠(본죽)  2017-01-02  2023-07-04
4  불법주정차  2017-01-01  15:33       서변동 행운보석  2017-01-02  2023-07-04
      구분        위반일자   위반시간          위반장소     과태료부과일자     데이터기준일자
0  불법주정차  2023-01-01  14:53     동천동 회전교차로  2023-01-02  2023-11-30
1  불법주정차  2023-01-01  14:59    노원LH아파트 후문  2023-01-02  2023-11-30
2  불법주정차  2023-01-01  14:58     노원한신더휴 입구  2023-01-02  2023-11-30
3  불법주정차  2023-01-01  14:57   대구일중학교 정문 앞  2023-01-02  2023-11-30
4  불법주정차  2023-01-01  14:53  오육민물수산 앞(칠성시  2023-01-02  2023-11-30
      구분        위반일자   위반시간            위반장소     과태료부과일자     데이터기준일자
0  불법주정차  2023-11-30  08:27      산격동 대학로 80  2023-12-01  2024-06-03
1  불법주정차  2023-11-30  09:08       팔달동 130-1  2023-12-01  2024-06-0

In [10]:
# 세 파일의 컬럼명이 같은지 확인합니다.
print("20230704 컬럼:", df_1.columns.tolist())
print("20231130 컬럼:", df_2.columns.tolist())
print("20240603 컬럼:", df_3.columns.tolist())

20230704 컬럼: ['구분', '위반일자', '위반시간', '위반장소', '과태료부과일자', '데이터기준일자']
20231130 컬럼: ['구분', '위반일자', '위반시간', '위반장소', '과태료부과일자', '데이터기준일자']
20240603 컬럼: ['구분', '위반일자', '위반시간', '위반장소', '과태료부과일자', '데이터기준일자']


In [11]:
# 각 파일의 위반일자 범위를 확인합니다.
for name, df in [("20230704", df_1), ("20231130", df_2), ("20240603", df_3)]:
    date_check = pd.to_datetime(df["위반일자"], errors="coerce")
    print(name, "위반일자 범위:", date_check.min(), "~", date_check.max())

20230704 위반일자 범위: 2017-01-01 00:00:00 ~ 2022-12-31 00:00:00
20231130 위반일자 범위: 2023-01-01 00:00:00 ~ 2023-11-29 00:00:00
20240603 위반일자 범위: 2023-11-30 00:00:00 ~ 2023-12-31 00:00:00


In [12]:
# 세 데이터프레임을 세로 방향으로 합칩니다.
df = pd.concat([df_1, df_2, df_3], ignore_index=True)

# 합친 뒤 전체 행과 열의 개수를 확인합니다.
print("통합 전처리 전 데이터 크기:", df.shape)

통합 전처리 전 데이터 크기: (681831, 6)


In [13]:
# '구분' 컬럼에 정상값인 '불법주정차' 이외의 값이 있는지 확인합니다.
print(df["구분"].value_counts(dropna=False))

# 구조가 깨진 행만 따로 확인합니다.
bad_rows = df[df["구분"] != "불법주정차"]
display(bad_rows)

구분
불법주정차                     681828
,2022-08-09,2023-07-04         1
,2022-08-10,2023-07-04         1
침산동 145-29 부근 횡단보도"            1
Name: count, dtype: int64


,구분,위반일자,위반시간,위반장소,과태료부과일자,데이터기준일자
532164,",2022-08-09,2023-07-04",NaN,NaN,NaN,NaN,NaN
532729,",2022-08-10,2023-07-04",NaN,NaN,NaN,NaN,NaN
537738,"침산동 145-29 부근 횡단보도""",2022-08-30,00:00,NaN,NaN,NaN


In [14]:
# '구분' 값이 '불법주정차'인 정상 행만 남깁니다.
df = df[df["구분"] == "불법주정차"].copy()

# 위반일자를 날짜형으로 변환합니다.
df["위반일자"] = pd.to_datetime(df["위반일자"], errors="coerce")

# 위반시간 앞뒤 공백을 제거합니다.
df["위반시간"] = df["위반시간"].astype(str).str.strip()

# 위반시간의 앞 두 자리를 숫자로 바꾸어 '시간' 컬럼을 만듭니다.
df["시간"] = pd.to_numeric(df["위반시간"].str[:2], errors="coerce")

In [15]:
# 날짜 분석에 사용할 연도와 월 컬럼을 만듭니다.
df["연도"] = df["위반일자"].dt.year
df["월"] = df["위반일자"].dt.month

# 숫자로 계산되는 요일을 한글 요일로 바꾸기 위한 딕셔너리입니다.
weekday_map = {
    0: "월요일",
    1: "화요일",
    2: "수요일",
    3: "목요일",
    4: "금요일",
    5: "토요일",
    6: "일요일"
}

# 위반일자에서 요일을 계산하여 새 컬럼을 만듭니다.
df["요일"] = df["위반일자"].dt.dayofweek.map(weekday_map)

In [16]:
# 주요 분석 컬럼에 결측치가 있는지 확인합니다.
print(df[["위반일자", "위반시간", "위반장소", "연도", "월", "요일", "시간"]].isnull().sum())

# 같은 값이 반복되는 행의 개수도 확인합니다.
# 이 데이터에는 차량번호나 고유 ID가 없기 때문에 같은 시각·장소의 여러 단속일 수 있어 삭제하지 않습니다.
print("완전히 동일한 행 수:", df.duplicated().sum())

위반일자    0
위반시간    0
위반장소    0
연도      0
월       0
요일      0
시간      0
dtype: int64
완전히 동일한 행 수: 122936


In [17]:
# 최종 전처리 결과를 확인합니다.
print("최종 데이터 크기:", df.shape)
print("위반일자 범위:", df["위반일자"].min(), "~", df["위반일자"].max())
print("연도별 행 수")
print(df.groupby("연도").size())

# 앞부분 5행을 확인합니다.
display(df.head())

최종 데이터 크기: (681828, 10)
위반일자 범위: 2017-01-01 00:00:00 ~ 2023-12-31 00:00:00
연도별 행 수
연도
2017     80211
2018     80917
2019    103086
2020    101209
2021    102151
2022    106172
2023    108082
dtype: int64


,구분,위반일자,위반시간,위반장소,과태료부과일자,데이터기준일자,시간,연도,월,요일
0,불법주정차,2017-01-01,11:08,태전1동 관음타운상가,2017-01-02,2023-07-04,11,2017,1,일요일
1,불법주정차,2017-01-01,12:42,대현동 동대구시장,2017-01-02,2023-07-04,12,2017,1,일요일
2,불법주정차,2017-01-01,14:24,침산동 건영하이츠(본죽),2017-01-02,2023-07-04,14,2017,1,일요일
3,불법주정차,2017-01-01,14:59,침산동 건영하이츠(본죽),2017-01-02,2023-07-04,14,2017,1,일요일
4,불법주정차,2017-01-01,15:33,서변동 행운보석,2017-01-02,2023-07-04,15,2017,1,일요일


In [18]:
# 전처리 결과를 저장할 폴더가 없으면 새로 만듭니다.
os.makedirs("data/processed", exist_ok=True)

# 팀원들이 공통으로 사용할 통합 데이터를 저장합니다.
# utf-8-sig로 저장하면 엑셀에서 열었을 때도 한글이 깨질 가능성이 적습니다.
output_path = "data/processed/violation_2017_2023.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("저장 완료:", output_path)

저장 완료: data/processed/violation_2017_2023.csv
